In [51]:
from google.colab import drive

In [52]:
import pandas as pd

In [53]:
import numpy as np

In [54]:
import gc

In [55]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
origem = '/content/drive/My Drive/API_2LOG/'

In [57]:
arquivo_geral = origem + 'Produto_Quimico.csv'

In [58]:
aq = pd.read_csv(arquivo_geral,low_memory=False,sep=';',encoding='UTF-8')

In [59]:
df_filtrado = aq[aq['Ano'].between(2013, 2025)]

In [60]:
print(df_filtrado.head())
print(f"Total de linhas após o filtro: {len(df_filtrado)}")

                 CNPJ                                       Razão Social  \
0  83.739.946/0001-00                     TRANSPORTADORA TRANSLEONE LTDA   
1  04.989.475/0001-03                                PURCOM QUIMICA LTDA   
2  76.104.397/0019-52                         TRANSPORTADORA SULISTA S/A   
3  46.804.539/0001-02  RCB IMPORTACAO E EXPORTACAO DE PRODUTOS QUIMIC...   
4  02.012.862/0164-06                             TAM LINHAS AEREAS S/A.   

           Estado  Município   Ano  \
0  SANTA CATARINA     ITAJAI  2024   
1       SAO PAULO    BARUERI  2024   
2  SANTA CATARINA  JOINVILLE  2024   
3       SAO PAULO  AMERICANA  2024   
4            PARA   SANTAREM  2024   

                                             Produto Quantidade Transportada  \
0                                        Óleo diesel                5.357,00   
1                            Produtos químicos, n.e.                   20,00   
2  Tintas e vernizes não especificados para usos ...                6.598,

In [61]:
import os

path_to_check = '/content/drive/My Drive/API_2LOG/'

if os.path.exists(path_to_check):
    print(f"Contents of {path_to_check}:")
    for item in os.listdir(path_to_check):
        print(item)
else:
    print(f"The directory {path_to_check} does not exist.")

Contents of /content/drive/My Drive/API_2LOG/:
Produto_Quimico.csv


In [62]:
aq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3925176 entries, 0 to 3925175
Data columns (total 16 columns):
 #   Column                   Dtype 
---  ------                   ----- 
 0   CNPJ                     object
 1   Razão Social             object
 2   Estado                   object
 3   Município                object
 4   Ano                      int64 
 5   Produto                  object
 6   Quantidade Transportada  object
 7   Unidade de Medida        object
 8   Tipo de Transporte       object
 9   Tipo de Armazenamento    object
 10  Plano de Emergência      object
 11  UF - origem              object
 12  Município - origem       object
 13  UF - destino             object
 14  Município - destino      object
 15  Situação Cadastral       object
dtypes: int64(1), object(15)
memory usage: 479.1+ MB


In [63]:
gc.collect()

0

In [64]:
# 1. Configurações
arquivo_entrada = arquivo_geral # COLOQUE O NOME DO SEU ARQUIVO AQUI
termos = ['óleo', 'diesel', 'gasolina', 'ácido', 'solução', 'água', 'combustível', 'etanol', 'lubrificante', 'gás', 'glp', 'oxigênio']
regex_liq = '|'.join(termos)

# 2. Processar em pedaços (Chunks) de 500 mil linhas por vez
# Isso impede que a RAM estoure
chunk_size = 500000
primeira_vez = True

for chunk in pd.read_csv(arquivo_entrada, sep=';', chunksize=chunk_size):

    # Filtrar anos logo no carregamento do pedaço
    chunk = chunk[chunk['Ano'].between(2013, 2025)].copy()

    if not chunk.empty:
        # Converter quantidade para número
        chunk['Quantidade Transportada'] = pd.to_numeric(
            chunk['Quantidade Transportada'].astype(str).str.replace(',', '.'),
            errors='coerce'
        ).fillna(0)

        # Identificar líquidos no pedaço
        is_liq = chunk['Produto'].str.lower().str.contains(regex_liq, na=False)

        # Criar a coluna convertida
        chunk['Quantidade_Final'] = chunk['Quantidade Transportada']

        # 1. Converte Tonelada para Litro (x1000)
        cond_t = is_liq & (chunk['Unidade de Medida'].str.lower().isin(['t', 'tonelada']))
        chunk.loc[cond_t, 'Quantidade_Final'] = chunk['Quantidade Transportada'] * 1000

        # 2. Converte Metro Cúbico para Litro (x1000)
        cond_m3 = is_liq & (chunk['Unidade de Medida'].str.lower().str.contains('metro|m3', na=False))
        chunk.loc[cond_m3, 'Quantidade_Final'] = chunk['Quantidade Transportada'] * 1000

        # 3. Salvar (append) nos arquivos finais
        modo = 'w' if primeira_vez else 'a'
        header = primeira_vez

        chunk[~is_liq].to_csv('solidos.csv', mode=modo, index=False, header=header, sep=';')
        chunk[is_liq].to_csv('liquidos_gasosos.csv', mode=modo, index=False, header=header, sep=';')

        primeira_vez = False

    # Limpar RAM após cada pedaço
    del chunk
    gc.collect()

print("Processamento concluído! Verifique os arquivos solidos.csv e liquidos_gasosos.csv na aba lateral.")

Processamento concluído! Verifique os arquivos solidos.csv e liquidos_gasosos.csv na aba lateral.


In [65]:
import pandas as pd
import numpy as np
import gc

# 1. Configurações
arquivo_entrada = arquivo_geral # COLOQUE O NOME DO SEU ARQUIVO AQUI
# Adicionei 'tinta', 'solvente' e 'resina' para melhorar o filtro
termos = ['óleo', 'diesel', 'gasolina', 'ácido', 'solução', 'água', 'combustível', 'etanol', 'lubrificante', 'gás', 'glp', 'oxigênio', 'tinta', 'solvente', 'resina']
regex_liq = '|'.join(termos)

chunk_size = 500000
primeira_vez = True

try:
    for chunk in pd.read_csv(arquivo_entrada, sep=';', chunksize=chunk_size, on_bad_lines='skip', low_memory=False):

        # Filtro de Ano
        chunk = chunk[chunk['Ano'].between(2013, 2025)].copy()

        if not chunk.empty:
            # Limpeza da quantidade
            chunk['Quantidade Transportada'] = pd.to_numeric(
                chunk['Quantidade Transportada'].astype(str).str.replace(',', '.'),
                errors='coerce'
            ).fillna(0)

            # Identificação
            is_liq = chunk['Produto'].str.lower().str.contains(regex_liq, na=False)

            # Criar a coluna de destino (Quantidade_Final)
            chunk['Quantidade_Final'] = chunk['Quantidade Transportada']

            # --- REGRAS DE CONVERSÃO ---

            # Regra A: Se for Líquido e estiver em Tonelada (t) -> x 1000
            mask_t = is_liq & (chunk['Unidade de Medida'].str.lower().isin(['t', 'tonelada']))
            chunk.loc[mask_t, 'Quantidade_Final'] = chunk['Quantidade Transportada'] * 1000

            # Regra B: Se for Líquido e estiver em Metro Cúbico (m3) -> x 1000
            mask_m3 = is_liq & (chunk['Unidade de Medida'].str.lower().str.contains('metro|m3', na=False))
            chunk.loc[mask_m3, 'Quantidade_Final'] = chunk['Quantidade Transportada'] * 1000

            # ---------------------------

            # Salva os resultados (modo 'a' anexa os dados aos arquivos)
            modo = 'w' if primeira_vez else 'a'
            header = primeira_vez

            chunk[~is_liq].to_csv('solidos.csv', mode=modo, index=False, header=header, sep=';')
            chunk[is_liq].to_csv('liquidos_gasosos.csv', mode=modo, index=False, header=header, sep=';')

            primeira_vez = False

        del chunk
        gc.collect()

    print("Processamento finalizado com sucesso!")

except FileNotFoundError:
    print(f"Erro: O arquivo '{arquivo_entrada}' não foi encontrado.")

Processamento finalizado com sucesso!


In [66]:
import pandas as pd

# 1. Carregar amostras maiores para validar
df_sol = pd.read_csv('solidos.csv', sep=';', nrows=1000)
df_liq = pd.read_csv('liquidos_gasosos.csv', sep=';', nrows=1000)

print("--- TESTE 1: FILTRO DE ANOS ---")
print(f"Anos encontrados nos sólidos: {df_sol['Ano'].unique()}")
print(f"Anos encontrados nos líquidos: {df_liq['Ano'].unique()}")

print("\n--- TESTE 2: CONVERSÃO DE UNIDADES (Líquidos) ---")
# Filtrar apenas quem era tonelada ou metro cúbico na amostra
checagem = df_liq[df_liq['Unidade de Medida'].str.lower().str.contains('tonelada|t|metro|m3', na=False)]

if not checagem.empty:
    # Se a Quantidade_Final for maior que a Transportada, a conta funcionou
    checagem['Sucesso'] = checagem['Quantidade_Final'] > checagem['Quantidade Transportada']
    print(checagem[['Produto', 'Unidade de Medida', 'Quantidade Transportada', 'Quantidade_Final']].head(10))
    print(f"\nConversões aplicadas com sucesso na amostra: {checagem['Sucesso'].sum()}")
else:
    print("Nenhum item de tonelada/m3 encontrado nas primeiras 1000 linhas para validar.")

print("\n--- TESTE 3: SEPARAÇÃO DE CATEGORIAS ---")
print(f"Exemplos no arquivo de SÓLIDOS: {df_sol['Produto'].unique()[:5]}")
print(f"Exemplos no arquivo de LÍQUIDOS: {df_liq['Produto'].unique()[:5]}")

--- TESTE 1: FILTRO DE ANOS ---
Anos encontrados nos sólidos: [2024 2023 2019 2022 2020 2021]
Anos encontrados nos líquidos: [2024 2023 2022 2021 2013 2014 2020 2016 2015]

--- TESTE 2: CONVERSÃO DE UNIDADES (Líquidos) ---
                                              Produto Unidade de Medida  \
0                                         Óleo diesel      Metro Cúbico   
1   Tintas e vernizes não especificados para usos ...             Litro   
2                                     Ácido sulfúrico             Litro   
3                                         Óleo diesel             Litro   
5                                            Oxigênio      Metro Cúbico   
7                                         Óleo diesel             Litro   
8                       Gasolina automotiva formulada             Litro   
10                              Óleo diesel formulado             Litro   
11                          Outros óleos combustíveis             Litro   
12  Gasolina automotiva mis

/tmp/ipykernel_9659/571906579.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  checagem['Sucesso'] = checagem['Quantidade_Final'] > checagem['Quantidade Transportada']
